# 第 09 章：多智能体协作与质检 实战

本章重点：
1. 如何将子网络编译 (SubGraph) 用作主网络节点，实现 Supervisor 模式。
2. (可选) 如何借助 LangSmith Evaluation 跑测数据集。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 统一实例化，支持重试策略配置
llm = init_chat_model(
    model="deepseek-chat",
    model_provider="openai",
    base_url="https://api.deepseek.com",
    streaming=True
)
print("LLM 服务准备就绪！")

## 1. SubGraph: 组长带小兵的开发模式
我们将建立一个翻译部门（组长 + 英翻小兵 + 日翻小兵）。

In [ ]:
from typing import TypedDict, Annotated, Sequence, Literal
import operator
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

# ---------------------
# 1. 定义共用状态
# ---------------------
class TranslationState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    target_lang: str
    final_result: str

# ---------------------
# 2. 开发子部门：英语专员
# ---------------------
def en_translator(state: TranslationState):
    msgs = [SystemMessage(content="你是专业英语翻译，请把用户的话翻译为英语。不要废话。直接给结果。")] + list(state["messages"])
    res = llm.invoke(msgs)
    return {"final_result": res.content}

en_builder = StateGraph(TranslationState)
en_builder.add_node("en_trans_node", en_translator)
en_builder.add_edge(START, "en_trans_node")
en_builder.add_edge("en_trans_node", END)
en_graph = en_builder.compile()

# ---------------------
# 3. 开发子部门：日语专员
# ---------------------
def jp_translator(state: TranslationState):
    msgs = [SystemMessage(content="你是专业日语翻译，请把用户的话翻译为日语。不要废话。直接给结果。")] + list(state["messages"])
    res = llm.invoke(msgs)
    return {"final_result": res.content}

jp_builder = StateGraph(TranslationState)
jp_builder.add_node("jp_trans_node", jp_translator)
jp_builder.add_edge(START, "jp_trans_node")
jp_builder.add_edge("jp_trans_node", END)
jp_graph = jp_builder.compile()

# ---------------------
# 4. 组装总公司部门 (Supervisor)
# ---------------------
def supervisor_router(state: TranslationState) -> Literal["en_dept", "jp_dept"]:
    if state.get("target_lang", "en") == "jp":
        return "jp_dept"
    return "en_dept"

main_builder = StateGraph(TranslationState)
# 【亮点】直接把 compiled_graph 当作节点加进去！
main_builder.add_node("en_dept", en_graph)
main_builder.add_node("jp_dept", jp_graph)

main_builder.add_conditional_edges(START, supervisor_router)
main_builder.add_edge("en_dept", END)
main_builder.add_edge("jp_dept", END)

multi_agent_app = main_builder.compile()
print("多智能体系统编译完成！组长已就位。")

### 执行多智能体路由测试

In [ ]:
query = "你好，很高兴认识你。我要订一张去东京的机票。"
test_state = {
    "messages": [HumanMessage(content=query)],
    "target_lang": "jp"
}

print("--- 提交到日文部门 ---")
for event in multi_agent_app.stream(test_state, stream_mode="values"):
    # 因为子图也会一层层把数据冒泡上来，我们可以看到执行链路
    if "final_result" in event:
        print("当前结果更新: ", event["final_result"])

print("\n--- 最终输出 ---")
final_state = multi_agent_app.invoke({
    "messages": [HumanMessage(content="The weather is nice.")],
    "target_lang": "en"
})
print(final_state["final_result"])

## 2. 质检与部署 (Conceptual Demo)

这里我们展示如何将其接入 LangSmith 服务。需要你配置好 `LANGCHAIN_API_KEY` 和 `LANGCHAIN_TRACING_V2=true`。
如果是本地不具备环境，此段代码仅作观赏与加深心智模型使用。

In [ ]:
import os
from langsmith import Client
from langchain.smith import RunEvalConfig, run_on_dataset

def optional_eval_demo():
    if not os.getenv("LANGCHAIN_API_KEY"):
        print("未配置 LangSmith 密钥，跳过此演示环节。")
        return

    client = Client()
    # 这里假设云端有一个叫做 "Translate-Test-Dataset" 的数据集
    eval_config = RunEvalConfig(
        evaluators=["qa"],
        prediction_key="final_result"
    )
    
    # 真正的评测调用
    # run_on_dataset(
    #    client=client,
    #    dataset_name="Translate-Test-Dataset",
    #    llm_or_chain_factory=multi_agent_app,
    #    evaluation=eval_config,
    # )
    print("质检脚本准备就绪，待数据集构建完成即可云端开跑。")

optional_eval_demo()